# 回収率の探索的分析（人気別）

本プロジェクトの評価軸は的中精度ではなく **回収率**（目標 110%）。
ここではモデルを使わない素朴なベースライン ―― 「人気順に機械的に賭けたら回収率はどうなるか」を確認し、
市場（オッズ）にどれだけの歪みがあるかを把握する。

学習・バックテストの本実装は `predictor/predictor/evaluation.py` 側にあるので、
ここでの結果は仮説出し・可視化用の探索にとどめる。
詳細な検討は `docs/plan/prediction-accuracy-followup.md` を参照。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from analysis.db import query_df
from analysis.plotting import setup_japanese_font

setup_japanese_font()
pd.set_option("display.max_columns", None)


def parse_yen(series: pd.Series) -> pd.Series:
    """"1,310" のようなカンマ区切り文字列を float に変換する（欠損は 0 = 不的中扱い）。"""
    return pd.to_numeric(series.str.replace(",", "", regex=False), errors="coerce").fillna(0)

## 単勝: 人気別の回収率

全レースで「1番人気に100円」「2番人気に100円」…を賭け続けた場合の回収率。

In [ ]:
win_df = query_df(
    """
    SELECT r.race_id, r.horse_number, r.popularity, p.payout
    FROM race_results r
    LEFT JOIN payoffs p
        ON p.race_id = r.race_id
        AND p.bet_type = '単勝'
        AND p.combination = r.horse_number
    WHERE r.popularity IS NOT NULL AND r.popularity ~ '^[0-9]+$'
    """
)
win_df["popularity"] = win_df["popularity"].astype(int)
win_df["payout"] = parse_yen(win_df["payout"])

win_summary = (
    win_df.groupby("popularity")
    .agg(n=("race_id", "size"), total_payout=("payout", "sum"))
    .assign(
        total_bet=lambda d: d["n"] * 100,
        recovery_rate=lambda d: (d["total_payout"] / d["total_bet"] * 100).round(1),
    )
)
win_summary = win_summary[win_summary["n"] >= 30]  # サンプル数が少ない人気帯（大穴側の欠測含む）は除外
win_summary

In [ ]:
ax = win_summary["recovery_rate"].plot.bar(figsize=(12, 4))
ax.axhline(100, color="red", linestyle="--", linewidth=1, label="収支分岐点 (100%)")
ax.axhline(110, color="green", linestyle="--", linewidth=1, label="目標 (110%)")
ax.set_ylabel("回収率 (%)")
ax.set_title("単勝: 人気別回収率")
ax.legend()
plt.tight_layout()

## 複勝: 人気別の回収率

In [ ]:
place_df = query_df(
    """
    SELECT r.race_id, r.horse_number, r.popularity, p.payout
    FROM race_results r
    LEFT JOIN payoffs p
        ON p.race_id = r.race_id
        AND p.bet_type = '複勝'
        AND p.combination = r.horse_number
    WHERE r.popularity IS NOT NULL AND r.popularity ~ '^[0-9]+$'
    """
)
place_df["popularity"] = place_df["popularity"].astype(int)
place_df["payout"] = parse_yen(place_df["payout"])

place_summary = (
    place_df.groupby("popularity")
    .agg(n=("race_id", "size"), total_payout=("payout", "sum"))
    .assign(
        total_bet=lambda d: d["n"] * 100,
        recovery_rate=lambda d: (d["total_payout"] / d["total_bet"] * 100).round(1),
    )
)
place_summary = place_summary[place_summary["n"] >= 30]
place_summary

## メモ

- 一般に単勝は上位人気ほど回収率が高く、下位人気ほど控除率（約20%）の影響で回収率が下がりやすい
  （＝市場は概ね効率的）。ここから外れる人気帯・条件があれば、モデルで狙うべきエッジの候補になる。
- `predicted_rank` や `win_prob` など予測結果を使った EV ベースの検証は、
  `predictor` の学習済みモデル出力（`output/prediction_*.csv` や `evaluation.py`）を読み込んで
  同様に人気帯 × EV 閾値でグリッド集計すると比較しやすい。